In [ ]:
import os
import xarray as xr
from collections import defaultdict

def inspect_merra2(directory="."):
    files = [f for f in os.listdir(directory) if f.endswith('.nc')]
    if not files:
        print("Error: No .nc files found in the directory.")
        return

    # 1. 分类文件
    streams = defaultdict(list)
    for f in files:
        # 假设文件名格式为: MERRA2_XXX.inst6_3d_ana_Np.YYYYMMDD.SUB.nc
        parts = f.split('.')
        if len(parts) > 0:
            stream_part = parts[0] # 获取 MERRA2_XXX
            streams[stream_part].append(f)

    print(f"{'Stream Type':<15} | {'File Count':<10}")
    print("-" * 30)
    for s in sorted(streams.keys()):
        print(f"{s:<15} | {len(streams[s]):<10}")

    print("\n" + "="*50)
    print("Checking internal structure for each stream...")
    print("="*50)

    # 2. 检查每种类型的一个代表性文件
    for s in sorted(streams.keys()):
        sample_file = streams[s][0]
        print(f"\n>>> Checking Stream: {s} (Sample: {sample_file})")
        try:
            # 使用 xarray 读取元数据
            ds = xr.open_dataset(os.path.join(directory, sample_file), decode_times=True)
            
            # 打印关键维度
            dims = dict(ds.dims)
            print(f"   Dimensions: {dims}")
            
            # 打印变量列表 (排除坐标变量)
            vars_list = [v for v in ds.data_vars]
            print(f"   Variables:  {vars_list}")
            
            # 检查时间步长（确认是否为日平均）
            if 'time' in ds.coords:
                time_size = len(ds.time)
                print(f"   Time steps: {time_size} (Expected 1 for daily average)")
            
            ds.close()
        except Exception as e:
            print(f"   Error reading {sample_file}: {e}")

if __name__ == "__main__":
    # 如果你的脚本不在数据目录，请修改下面的路径
    inspect_merra2("/mnt/soclim0/public_data/weiji/MERRA2M2I6NPANA")

In [ ]:
import os
import pandas as pd

def check_fixed_path():
    # 强制指定数据存放路径
    target_path = "/mnt/soclim0/public_data/weiji/MERRA2M2I6NPANA"
    
    # 1. 检查目录是否存在并读取文件
    if not os.path.exists(target_path):
        print(f"❌ 错误：路径 {target_path} 不存在！")
        return
    
    # 扫描所有 .nc 文件
    all_files = [f for f in os.listdir(target_path) if f.endswith('.nc')]
    print(f"--- 目录扫描结果 ---")
    print(f"目标路径: {target_path}")
    print(f"找到的 .nc 文件总数: {len(all_files)}")
    
    if len(all_files) == 0:
        print("⚠️ 警告：该路径下没有找到任何 .nc 文件，请确认路径是否正确。")
        return

    # 2. 生成预期的日期列表
    start_date = "1980-01-01"
    end_date = "2026-02-01"
    expected_dates = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
    
    # 3. 核心比对逻辑 (只要日期字符串在文件名里就算匹配成功)
    # 为了效率，把所有文件名拼成一个巨大的长字符串
    filenames_blob = "|".join(all_files)
    
    actual_count = 0
    missing_dates = []
    
    for date in expected_dates:
        if date in filenames_blob:
            actual_count += 1
        else:
            missing_dates.append(date)

    # 4. 报告
    print(f"\n--- 完整性核实报告 ---")
    print(f"预期总天数: {len(expected_dates)}")
    print(f"匹配成功数: {actual_count}")
    print(f"缺失总天数: {len(missing_dates)}")
    
    if not missing_dates:
        print("\n✅ 状态: 数据完美，100% 完整！")
    else:
        print(f"\n❌ 状态: 存在缺失。")
        historical = [d for d in missing_dates if d < "20260101"]
        recent = [d for d in missing_dates if d >= "20260101"]
        
        if historical:
            print(f"⚠️ 历史数据断档 (需补下): {historical[:5]} ... (共{len(historical)}天)")
        if recent:
            print(f"ℹ️ 2026年待发布数据: {recent}")

        # 将缺失日期保存，方便后续操作
        with open("dates_to_fix.txt", "w") as f:
            for d in missing_dates:
                f.write(f"{d}\n")
        print("\n详细缺失清单已保存至当前目录下的: dates_to_fix.txt")

if __name__ == "__main__":
    check_fixed_path()

In [ ]:
import os
import re

# 数据路径
src_dir = "/mnt/soclim0/public_data/weiji/MERRA2M2I6NPANA"

# 获取前5个文件作为样本
files = [f for f in os.listdir(src_dir) if f.endswith('.nc')][:10]

print("--- 文件名解析调试 ---")
for f in files:
    # 尝试匹配 19xx 或 20xx 开头的 8 位数字
    match = re.search(r'\.(19\d{2}|20\d{2})\d{4}\.', f)
    if match:
        year = match.group(1)
        print(f"文件名: {f}  -->  解析出的年份: {year}")
    else:
        # 如果上面的正则失败，看看直接提取所有数字
        all_nums = re.findall(r'\d+', f)
        print(f"文件名: {f}  -->  未匹配！提取到的数字序列: {all_nums}")

print("\n--- 预想的年份提取逻辑 ---")
# 模拟 Shell 中的年份探测
all_files_str = "\n".join(os.listdir(src_dir))
# 匹配 .YYYYMMDD. 结构中的前四位
years = sorted(list(set(re.findall(r'\.(19\d{2}|20\d{2})\d{4}\.', all_filenames_str := "\n".join(os.listdir(src_dir))))))
print(f"探测到的唯一年份列表: {years}")

In [ ]:
import xarray as xr
import os

def inspect_details(file_path):
    ds = xr.open_dataset(file_path, decode_times=False)
    
    print(f"检查文件: {os.path.basename(file_path)}")
    print("-" * 50)
    
    # 1. 检查维度
    print(f"维度 (Dimensions): {dict(ds.sizes)}")
    
    # 2. 检查变量
    print(f"变量 (Variables): {list(ds.data_vars)}")
    
    # 3. 检查气压层 (lev)
    if 'lev' in ds.coords:
        levs = ds.lev.values
        print(f"气压层数: {len(levs)}")
        print(f"层级数值 (前3层): {levs[:3]}")
        print(f"层级数值 (后3层): {levs[-3:]}")
        # 检查是否单调递增或递减
        is_monot = all(levs[i] < levs[i+1] for i in range(len(levs)-1)) or \
                   all(levs[i] > levs[i+1] for i in range(len(levs)-1))
        print(f"气压层是否单调排列: {'是' if is_monot else '否 (警告: 需要重排)'}")

    # 4. 检查坐标范围
    print(f"经度范围: {ds.lon.min().values} 到 {ds.lon.max().values}")
    print(f"纬度范围: {ds.lat.min().values} 到 {ds.lat.max().values}")
    
    ds.close()

# 随便挑两个不同时期的文件对比
src_dir = "/mnt/soclim0/public_data/weiji/MERRA2M2I6NPANA"
sample_files = [
    os.path.join(src_dir, "MERRA2_100.inst6_3d_ana_Np.19820410.SUB.nc"),
    os.path.join(src_dir, "MERRA2_401.inst6_3d_ana_Np.20210930.SUB.nc")
]

for f in sample_files:
    if os.path.exists(f):
        inspect_details(f)
        print("\n")

In [ ]:
# ============================================================
# DEBUG ONLY: inspect raw daily SUB files for one year
# 目的：为后续 CDO + sh 提供可靠信息，不做正式处理
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr

SRC_DIR = "/mnt/soclim0/public_data/weiji/MERRA2M2I6NPANA"
YEAR = 1981
VARS_TO_CHECK = ["O3", "T", "U", "V"]

# 抽样查看哪些日文件（前3、中间3、后3）
N_SAMPLE_HEAD = 3
N_SAMPLE_TAIL = 3
CHECK_MIDDLE = True


def parse_date_from_filename(fp):
    m = re.search(r"\.(\d{8})\.SUB\.nc$", os.path.basename(fp))
    if not m:
        raise ValueError(f"Cannot parse date from filename: {fp}")
    return pd.to_datetime(m.group(1), format="%Y%m%d")


def choose_sample_indices(n):
    idx = list(range(min(N_SAMPLE_HEAD, n)))
    if CHECK_MIDDLE and n > 6:
        mid = n // 2
        idx += [max(0, mid-1), mid, min(n-1, mid+1)]
    tail_start = max(0, n - N_SAMPLE_TAIL)
    idx += list(range(tail_start, n))
    return sorted(set(idx))


def print_header(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def summarize_array(name, arr, max_items=8):
    arr = np.asarray(arr)
    print(f"{name}: dtype={arr.dtype}, shape={arr.shape}")
    if arr.ndim == 1:
        if len(arr) <= max_items * 2:
            print(arr)
        else:
            print("head:", arr[:max_items])
            print("tail:", arr[-max_items:])
    else:
        print(arr)


# ------------------------------------------------------------
# 1) Find files and inspect filename-based dates
# ------------------------------------------------------------
files = sorted(glob.glob(os.path.join(SRC_DIR, f"MERRA2_*.inst6_3d_ana_Np.{YEAR}????.SUB.nc")))
if not files:
    raise FileNotFoundError(f"No files found for year {YEAR} in {SRC_DIR}")

print_header(f"File discovery for YEAR={YEAR}")
print(f"Found {len(files)} files")

dates = pd.DatetimeIndex([parse_date_from_filename(f) for f in files])
print("First 5 dates from filenames:", dates[:5].tolist())
print("Last  5 dates from filenames:", dates[-5:].tolist())

expected = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq="D")
print(f"Expected daily length: {len(expected)}")
print(f"Parsed   daily length: {len(dates)}")
print(f"Exact match with complete daily calendar? {dates.equals(expected)}")

missing = expected.difference(dates)
extra = dates.difference(expected)

print(f"Missing dates count: {len(missing)}")
if len(missing) > 0:
    print("Missing example:", missing[:10].tolist())

print(f"Extra dates count: {len(extra)}")
if len(extra) > 0:
    print("Extra example:", extra[:10].tolist())

# 是否有重复
dup_mask = dates.duplicated()
print(f"Duplicated filename dates? {dup_mask.any()}")
if dup_mask.any():
    print("Duplicated examples:", dates[dup_mask][:10].tolist())

sample_idx = choose_sample_indices(len(files))


# ------------------------------------------------------------
# 2) Inspect sampled raw files
# ------------------------------------------------------------
for i in sample_idx:
    fp = files[i]
    file_date = parse_date_from_filename(fp)

    print_header(f"Sample file #{i}: {os.path.basename(fp)}")
    print(f"Filename date = {file_date}")

    # ---- raw numeric time ----
    with xr.open_dataset(fp, decode_times=False) as ds_raw:
        print("Dimensions:", dict(ds_raw.sizes))
        print("Data vars :", list(ds_raw.data_vars))

        if "time" in ds_raw:
            t_raw = ds_raw["time"]
            print("\n[time raw attrs]")
            print(dict(t_raw.attrs))
            summarize_array("time raw values", t_raw.values)

        if "time_bnds" in ds_raw:
            tb_raw = ds_raw["time_bnds"]
            print("\n[time_bnds raw attrs]")
            print(dict(tb_raw.attrs))
            summarize_array("time_bnds raw values", tb_raw.values)

        for v in VARS_TO_CHECK:
            if v in ds_raw:
                print(f"\n[{v} attrs]")
                print(dict(ds_raw[v].attrs))
                print(f"{v} dims={ds_raw[v].dims}, shape={ds_raw[v].shape}")

        if "lev" in ds_raw.coords:
            print("\n[lev attrs]")
            print(dict(ds_raw["lev"].attrs))
            summarize_array("lev", ds_raw["lev"].values)

    # ---- decoded time ----
    with xr.open_dataset(fp, decode_times=True) as ds_dec:
        if "time" in ds_dec:
            print("\n[time decoded]")
            summarize_array("time decoded values", ds_dec["time"].values)

        if "time_bnds" in ds_dec:
            print("\n[time_bnds decoded]")
            summarize_array("time_bnds decoded values", ds_dec["time_bnds"].values)


# ------------------------------------------------------------
# 3) One-line interpretation for shell/CDO design
# ------------------------------------------------------------
print_header("Interpretation for downstream CDO + sh")

print("Rule 1: The most trustworthy date source is the FILENAME (YYYYMMDD).")
print("Rule 2: In these SUB files, raw time is often just a local offset like 540 minutes.")
print("Rule 3: If each daily file has its own 'units since YYYY-MM-DD 00:00:00',")
print("        then direct ncrcat can destroy the calendar sequence.")
print("Rule 4: Before writing the final yearly files, verify that merged timestamps")
print("        become a continuous daily axis.")
print("Rule 5: For final time axis, filename-derived dates are safest.")

# 建议给 shell 脚本使用的目标轴（如果你想固定到 09:00）
time_target = dates + pd.Timedelta(hours=9)
print("\nSuggested canonical timestamps from filenames:")
print("First 5:", time_target[:5].tolist())
print("Last  5:", time_target[-5:].tolist())

base = pd.Timestamp("1980-01-01 00:00:00")
days_since = (time_target - base) / pd.Timedelta(days=1)
print("\nSuggested numeric encoding if needed:")
print("units = 'days since 1980-01-01 00:00:00'")
print("First 5 numeric values:", days_since[:5].astype(float).tolist())
print("Last  5 numeric values:", days_since[-5:].astype(float).tolist())

In [ ]:
# ============================================================
# DEBUG: inspect processed yearly MERRA2 files
# 只做读取检查，不修改任何文件
# ============================================================

import os
import numpy as np
import pandas as pd
import xarray as xr

ROOT = "/mnt/soclim0/public_data/weiji/MERRA2_Processed_FixedTime"
YEAR = 2026
VARS = ["U", "V", "T", "O3"]

def print_header(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

def summarize_time(da):
    t = pd.to_datetime(da.time.values)
    print(f"time length      : {len(t)}")
    print(f"first time       : {t[0]}")
    print(f"last time        : {t[-1]}")
    if len(t) > 1:
        dt_days = np.diff(t.values).astype("timedelta64[s]").astype(float) / 86400.0
        uniq = np.unique(np.round(dt_days, 6))
        print(f"unique dt (days) : {uniq}")
    print(f"all unique?      : {pd.Index(t).is_unique}")

def expected_daily_axis(year, n):
    # 从该年1月1日开始，长度 n 的 daily 序列
    return pd.date_range(f"{year:04d}-01-01", periods=n, freq="D")

for var in VARS:
    fp = os.path.join(ROOT, var, f"MERRA2.{var}.{YEAR}.nc")
    print_header(f"{var} | {fp}")

    if not os.path.exists(fp):
        print("File not found.")
        continue

    ds = xr.open_dataset(fp)
    print(ds)

    # 找主变量
    if var not in ds.data_vars:
        print(f"Main variable {var} not found in dataset.")
        ds.close()
        continue

    da = ds[var]

    print("\n[dataset attrs]")
    print(dict(ds.attrs))

    print("\n[variable attrs]")
    print(dict(da.attrs))

    print("\n[dims / shape]")
    print(f"dims  : {da.dims}")
    print(f"shape : {da.shape}")

    # 时间轴检查
    if "time" in da.coords:
        print("\n[time summary]")
        summarize_time(da)

        t = pd.to_datetime(da.time.values)
        expected = expected_daily_axis(YEAR, len(t))
        print(f"matches expected daily axis from {YEAR}-01-01? {pd.Index(t).equals(pd.Index(expected))}")

    # 坐标检查
    for coord in ["lev", "lat", "lon"]:
        if coord in da.coords:
            vals = da[coord].values
            print(f"\n[{coord}]")
            print(f"shape: {vals.shape}")
            print(f"first: {vals[:8] if vals.ndim == 1 else vals}")
            print(f"last : {vals[-8:] if vals.ndim == 1 else vals}")
            print(f"attrs: {dict(da[coord].attrs)}")

    # 数值范围抽样检查
    print("\n[value range]")
    vals = da.isel(time=0).values
    vals = vals[np.isfinite(vals)]
    if vals.size > 0:
        q = np.nanpercentile(vals, [0, 1, 5, 25, 50, 75, 95, 99, 100])
        labs = ["min", "p01", "p05", "p25", "p50", "p75", "p95", "p99", "max"]
        for lab, qq in zip(labs, q):
            print(f"{lab:>4s}: {qq:.6e}")
    else:
        print("All values are NaN on first timestep.")

    ds.close()

In [ ]:
import os
import re
import pandas as pd
import xarray as xr

# ============================================================
# 0) 路径设置：只读 /mnt，只写 /home
# ============================================================
ROOT_DIR = "/mnt/backup_ETH/Marina/MERRA2"
OUT_DIR = "/home/weiji/restart_exam/code/20260307merra2data"

os.makedirs(OUT_DIR, exist_ok=True)

OUT_DIR_CSV   = os.path.join(OUT_DIR, "merra2_directory_inventory.csv")
OUT_GROUP_CSV = os.path.join(OUT_DIR, "merra2_nc_group_inventory.csv")
OUT_MISC_CSV  = os.path.join(OUT_DIR, "merra2_other_files_inventory.csv")

# ============================================================
# 1) 基础工具
# ============================================================
AUX_VAR_NAMES = {
    "bnds", "nbnds", "nv",
    "time_bnds", "lat_bnds", "lon_bnds", "lev_bnds",
    "latitude_bnds", "longitude_bnds"
}

def human_size(num_bytes):
    gb = num_bytes / (1024 ** 3)
    if gb >= 1:
        return f"{gb:.2f} GB"
    mb = num_bytes / (1024 ** 2)
    if mb >= 1:
        return f"{mb:.2f} MB"
    kb = num_bytes / 1024
    return f"{kb:.2f} KB"

def relpath_safe(path, root):
    return os.path.relpath(path, root)

def safe_get_attr(ds, key, default=""):
    return str(ds.attrs.get(key, default))

def safe_open_metadata(ncfile):
    # 只读 metadata，不 decode 时间，不触发大数组加载
    return xr.open_dataset(ncfile, decode_times=False)

# ============================================================
# 2) 变量、坐标、维度
# ============================================================
def classify_vars(ds):
    coord_names = set(ds.coords)
    data_var_names = list(ds.data_vars)

    core_vars = []
    aux_vars = []

    for v in data_var_names:
        lower = v.lower()

        if v in AUX_VAR_NAMES or lower.endswith("_bnds"):
            aux_vars.append(v)
            continue

        # 如果变量属性里显示它是 bounds，也归为辅助变量
        try:
            bounds_attr = str(ds[v].attrs.get("bounds", "")).strip()
        except Exception:
            bounds_attr = ""

        if bounds_attr:
            # 这里只保守处理，不因为有 bounds 属性就排除掉主变量
            core_vars.append(v)
        else:
            core_vars.append(v)

    return {
        "coords": ", ".join(coord_names),
        "core_vars": ", ".join(core_vars),
        "aux_vars": ", ".join(aux_vars),
    }

def get_dims(ds):
    return ", ".join([f"{k}={v}" for k, v in ds.sizes.items()])

# ============================================================
# 3) 分辨率与垂直信息
# ============================================================
def parse_dataresolution_string(s):
    """
    例如:
    '0.5 x 0.625 (42 pressure levels)'
    -> horizontal='0.5 x 0.625', vertical='42 pressure levels'
    """
    if not s:
        return "", ""

    s = str(s).strip()
    m = re.match(r"^\s*([^()]+?)\s*(?:\((.*?)\))?\s*$", s)
    if not m:
        return s, ""

    horizontal = (m.group(1) or "").strip()
    vertical = (m.group(2) or "").strip()
    return horizontal, vertical

def infer_horizontal_resolution(ds):
    # 1) 优先读 DataResolution
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        hres, _ = parse_dataresolution_string(data_res)
        if hres:
            return hres

    # 2) 再读经纬度分辨率属性
    lat_res = ds.attrs.get("LatitudeResolution", None)
    lon_res = ds.attrs.get("LongitudeResolution", None)
    if lat_res is not None and lon_res is not None:
        return f"{lat_res} x {lon_res}"

    # 3) 最后从坐标推断
    try:
        lat_name = None
        lon_name = None

        for cand in ["lat", "latitude", "y"]:
            if cand in ds.coords:
                lat_name = cand
                break

        for cand in ["lon", "longitude", "x"]:
            if cand in ds.coords:
                lon_name = cand
                break

        if lat_name and lon_name and ds[lat_name].size > 1 and ds[lon_name].size > 1:
            lat_vals = ds[lat_name].values
            lon_vals = ds[lon_name].values
            dlat = abs(float(lat_vals[1] - lat_vals[0]))
            dlon = abs(float(lon_vals[1] - lon_vals[0]))
            return f"{dlat} x {dlon}"
    except Exception:
        pass

    return ""

def infer_vertical_info(ds):
    # 1) 优先读 DataResolution 里的括号信息
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        _, vinfo = parse_dataresolution_string(data_res)
        if vinfo:
            return vinfo

    # 2) 再根据常见垂直坐标推断
    for zname in ["lev", "level", "plev", "isobaricInhPa", "isobaric", "pressure"]:
        if zname in ds.coords:
            try:
                nlev = ds[zname].size
                units = str(ds[zname].attrs.get("units", "")).strip()
                long_name = str(ds[zname].attrs.get("long_name", "")).strip()
                if units and long_name:
                    return f"{nlev} levels; {long_name}; {units}"
                elif units:
                    return f"{nlev} levels; {units}"
                else:
                    return f"{nlev} levels"
            except Exception:
                return "vertical coordinate present"

    return ""

# ============================================================
# 4) 时间信息：header
# ============================================================
def get_header_time_info(ds):
    row = {
        "header_temporal_range": "",
        "header_begin_date": "",
        "header_begin_time": "",
        "header_end_date": "",
        "header_end_time": "",
    }

    row["header_temporal_range"] = safe_get_attr(ds, "TemporalRange")
    row["header_begin_date"] = safe_get_attr(ds, "RangeBeginningDate")
    row["header_begin_time"] = safe_get_attr(ds, "RangeBeginningTime")
    row["header_end_date"] = safe_get_attr(ds, "RangeEndingDate")
    row["header_end_time"] = safe_get_attr(ds, "RangeEndingTime")

    return row

# ============================================================
# 5) 文件名时间 token 提取
#    只用于“收集信息”，不擅自推断精确日历意义
# ============================================================
DATE_PATTERNS = [
    # YYYY-MM-DD / YYYY_MM_DD / YYYY.MM.DD
    ("date10", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    # YYYYMMDD
    ("date8", re.compile(r"(?<!\d)((?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    # YYYY-MM / YYYY_MM / YYYY.MM
    ("yearmon", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2]))(?!\d)")),
    # YYYY
    ("year", re.compile(r"(?<!\d)((?:19|20)\d{2})(?!\d)")),
]

def normalize_time_token(token, kind):
    token = token.replace("_", "-").replace(".", "-")
    if kind == "date8":
        return f"{token[0:4]}-{token[4:6]}-{token[6:8]}"
    return token

def extract_time_tokens_no_overlap(text):
    """
    按优先级提取，避免 YYYY-MM 里面又重复抽出 YYYY
    """
    accepted = []
    occupied = []

    def overlaps(span):
        s1, e1 = span
        for s2, e2 in occupied:
            if not (e1 <= s2 or e2 <= s1):
                return True
        return False

    for kind, pattern in DATE_PATTERNS:
        for m in pattern.finditer(text):
            span = m.span(1)
            if overlaps(span):
                continue
            raw = m.group(1)
            norm = normalize_time_token(raw, kind)
            accepted.append((span[0], span[1], kind, raw, norm))
            occupied.append(span)

    accepted.sort(key=lambda x: x[0])
    return accepted

def extract_time_tokens_from_filename(fname):
    base = os.path.basename(fname)
    records = extract_time_tokens_no_overlap(base)
    return [x[4] for x in records]  # 返回规范化后的 token

# ============================================================
# 6) 文件名 pattern 归一化
#    先替换范围，再替换单点日期
# ============================================================
def normalize_filename_pattern(fname):
    base = os.path.basename(fname)

    # ---- 先替换范围型写法 ----
    # YYYY-MM.YYYY-MM
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<RANGE_YM_YM>",
        base
    )

    # YYYY-MM.YYYY
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_YM_Y>",
        base
    )

    # YYYY.YYYY
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )

    # YYYY-YYYY
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}-(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )

    # ---- 再替换单点日期 ----
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE10>",
        base
    )

    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE8>",
        base
    )

    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<YEARMON>",
        base
    )

    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?!\d)",
        "<YEAR>",
        base
    )

    return base

# ============================================================
# 7) sample file metadata 读取
# ============================================================
def summarize_sample_nc(ncfile):
    row = {
        "sample_status": "OK",
        "sample_title": "",
        "sample_short_name": "",
        "sample_long_name": "",
        "sample_frequency": "",
        "sample_doi": "",
        "sample_version": "",
        "sample_coords": "",
        "sample_core_vars": "",
        "sample_aux_vars": "",
        "sample_dimensions": "",
        "sample_horizontal_resolution": "",
        "sample_vertical_info": "",
        "header_temporal_range": "",
        "header_begin_date": "",
        "header_begin_time": "",
        "header_end_date": "",
        "header_end_time": "",
    }

    try:
        ds = safe_open_metadata(ncfile)

        var_info = classify_vars(ds)
        time_info = get_header_time_info(ds)

        row["sample_title"] = safe_get_attr(ds, "Title")
        row["sample_short_name"] = safe_get_attr(ds, "ShortName")
        row["sample_long_name"] = safe_get_attr(ds, "LongName")
        row["sample_frequency"] = safe_get_attr(ds, "frequency")
        row["sample_doi"] = safe_get_attr(ds, "identifier_product_doi")
        row["sample_version"] = safe_get_attr(ds, "VersionID")

        row["sample_coords"] = var_info["coords"]
        row["sample_core_vars"] = var_info["core_vars"]
        row["sample_aux_vars"] = var_info["aux_vars"]
        row["sample_dimensions"] = get_dims(ds)
        row["sample_horizontal_resolution"] = infer_horizontal_resolution(ds)
        row["sample_vertical_info"] = infer_vertical_info(ds)

        row.update(time_info)

        ds.close()

    except Exception as e:
        row["sample_status"] = f"FAILED: {e}"

    return row

# ============================================================
# 8) group 分类与时间来源判断
# ============================================================
def has_time_like_pattern(pattern):
    markers = [
        "<DATE10>", "<DATE8>", "<YEARMON>", "<YEAR>",
        "<RANGE_YM_YM>", "<RANGE_YM_Y>", "<RANGE_Y_Y>"
    ]
    return any(m in pattern for m in markers)

def infer_product_type(pattern, n_files):
    lower = pattern.lower()

    if n_files > 1 and has_time_like_pattern(pattern):
        return "raw_file_family"

    if n_files > 1 and not has_time_like_pattern(pattern):
        return "multi_file_family"

    # 单文件产品
    if ".zm" in lower or "_zm" in lower:
        return "processed_single_zm"
    if ".clim" in lower or "_clim" in lower:
        return "processed_single_clim"
    if ".pv" in lower or "_pv" in lower:
        return "processed_single_pv"
    if "_dm" in lower:
        return "processed_single_dm"

    return "single_product"

def infer_group_hint(pattern, n_files, sample_meta):
    lower = pattern.lower()
    tags = []

    tags.append(infer_product_type(pattern, n_files))

    if has_time_like_pattern(pattern):
        tags.append("time_in_filename")

    if ".zm" in lower or "_zm" in lower:
        tags.append("zonal_mean")
    if ".clim" in lower or "_clim" in lower:
        tags.append("climatology")
    if ".pv" in lower or "_pv" in lower:
        tags.append("PV_related")
    if "runmean" in lower:
        tags.append("running_mean")
    if "_dm" in lower:
        tags.append("daily_mean_or_dm")
    if "inst6" in lower:
        tags.append("inst6_source")
    if "noleap" in lower:
        tags.append("noleap")
    if "r144x96" in lower or "new_grid" in lower:
        tags.append("target_grid_or_regridded")

    freq = sample_meta.get("sample_frequency", "")
    if freq:
        tags.append(f"freq={freq}")

    return ", ".join(tags)

def choose_group_time_source(pattern, n_files, token_min, token_max, sample_meta):
    """
    这里只决定“后面该优先看哪类时间信息”
    不做最终人工整理。
    """
    header_range = sample_meta.get("header_temporal_range", "")
    has_tokens = bool(token_min or token_max)

    if n_files > 1 and has_time_like_pattern(pattern) and has_tokens:
        return "filename_range"

    if n_files == 1 and header_range:
        return "header_temporal_range"

    if n_files == 1 and (sample_meta.get("header_begin_date") or sample_meta.get("header_end_date")):
        return "header_begin_end"

    if has_tokens:
        return "filename_range"

    return "unknown"

# ============================================================
# 9) 扫描整个目录树
# ============================================================
dir_rows = []
group_rows = []
misc_rows = []

for current_dir, subdirs, files in os.walk(ROOT_DIR):
    subdirs.sort()
    files.sort()

    full_files = [os.path.join(current_dir, f) for f in files]

    nc_files = [f for f in full_files if f.endswith(".nc") or f.endswith(".nc4")]
    other_files = [f for f in full_files if not (f.endswith(".nc") or f.endswith(".nc4"))]

    # -------------------------
    # 目录级信息
    # -------------------------
    dir_rows.append({
        "directory": current_dir,
        "relative_directory": relpath_safe(current_dir, ROOT_DIR),
        "n_subdirs": len(subdirs),
        "subdirs": ", ".join(subdirs),
        "n_nc_files": len(nc_files),
        "n_other_files": len(other_files),
    })

    # -------------------------
    # 非 nc 文件
    # -------------------------
    for f in other_files:
        misc_rows.append({
            "directory": current_dir,
            "relative_directory": relpath_safe(current_dir, ROOT_DIR),
            "filename": os.path.basename(f),
            "size": human_size(os.path.getsize(f)),
            "extension": os.path.splitext(f)[1],
        })

    # -------------------------
    # 当前目录下的 nc 文件：按 pattern 分组
    # -------------------------
    if len(nc_files) == 0:
        continue

    groups = {}
    for f in nc_files:
        patt = normalize_filename_pattern(os.path.basename(f))
        groups.setdefault(patt, []).append(f)

    for patt, flist in sorted(groups.items(), key=lambda x: x[0]):
        flist = sorted(flist)
        sample_file = flist[0]

        sample_meta = summarize_sample_nc(sample_file)

        # 文件名时间 token 收集（对整个组）
        all_tokens = []
        for f in flist:
            all_tokens.extend(extract_time_tokens_from_filename(os.path.basename(f)))

        all_tokens = sorted(set(all_tokens))
        token_min = all_tokens[0] if len(all_tokens) > 0 else ""
        token_max = all_tokens[-1] if len(all_tokens) > 0 else ""

        # 计算组总大小
        total_size_bytes = sum(os.path.getsize(f) for f in flist)

        # 时间来源判定
        time_source = choose_group_time_source(
            pattern=patt,
            n_files=len(flist),
            token_min=token_min,
            token_max=token_max,
            sample_meta=sample_meta,
        )

        group_rows.append({
            "directory": current_dir,
            "relative_directory": relpath_safe(current_dir, ROOT_DIR),

            "file_pattern": patt,
            "n_files": len(flist),
            "total_size": human_size(total_size_bytes),

            "sample_file": sample_file,
            "first_file": os.path.basename(flist[0]),
            "last_file": os.path.basename(flist[-1]),

            # 文件名层面的时间信息
            "filename_token_min": token_min,
            "filename_token_max": token_max,

            # 只标明后面应该优先看哪类时间，不替你直接生成最终结论
            "time_coverage_source_hint": time_source,

            "product_type": infer_product_type(patt, len(flist)),
            "group_hint": infer_group_hint(patt, len(flist), sample_meta),

            # sample header 信息
            "sample_status": sample_meta["sample_status"],
            "sample_title": sample_meta["sample_title"],
            "sample_short_name": sample_meta["sample_short_name"],
            "sample_long_name": sample_meta["sample_long_name"],
            "sample_frequency": sample_meta["sample_frequency"],
            "sample_doi": sample_meta["sample_doi"],
            "sample_version": sample_meta["sample_version"],

            "sample_coords": sample_meta["sample_coords"],
            "sample_core_vars": sample_meta["sample_core_vars"],
            "sample_aux_vars": sample_meta["sample_aux_vars"],
            "sample_dimensions": sample_meta["sample_dimensions"],
            "sample_horizontal_resolution": sample_meta["sample_horizontal_resolution"],
            "sample_vertical_info": sample_meta["sample_vertical_info"],

            # header 时间信息保留原始字段，供后续人工判断
            "header_temporal_range": sample_meta["header_temporal_range"],
            "header_begin_date": sample_meta["header_begin_date"],
            "header_begin_time": sample_meta["header_begin_time"],
            "header_end_date": sample_meta["header_end_date"],
            "header_end_time": sample_meta["header_end_time"],
        })

# ============================================================
# 10) 保存原始 inventory（不是最终表格）
# ============================================================
df_dirs = pd.DataFrame(dir_rows)
df_groups = pd.DataFrame(group_rows)
df_misc = pd.DataFrame(misc_rows)

df_dirs.to_csv(OUT_DIR_CSV, index=False)
df_groups.to_csv(OUT_GROUP_CSV, index=False)
df_misc.to_csv(OUT_MISC_CSV, index=False)

# ============================================================
# 11) 简短终端输出
# ============================================================
print("\nDone.")
print(f"Directories collected : {len(df_dirs)}")
print(f"NC groups collected   : {len(df_groups)}")
print(f"Other files collected : {len(df_misc)}")

print("\nSaved raw inventories to:")
print(OUT_DIR_CSV)
print(OUT_GROUP_CSV)
print(OUT_MISC_CSV)

print("\nPreview of nc groups:")
preview_cols = [
    "relative_directory",
    "file_pattern",
    "n_files",
    "product_type",
    "filename_token_min",
    "filename_token_max",
    "time_coverage_source_hint",
    "sample_core_vars",
    "sample_aux_vars",
    "sample_horizontal_resolution",
    "sample_vertical_info",
    "sample_status",
]
if len(df_groups) > 0:
    print(df_groups[preview_cols].head(50).to_string(index=False))
else:
    print("No nc/nc4 files found.")

In [ ]:
import os
import re
import math
import pandas as pd
import xarray as xr

try:
    import cftime
except ImportError:
    cftime = None

# ============================================================
# 0) 路径设置：只读 /mnt，只写 /home
# ============================================================
ROOT_DIR = "/mnt/backup_ETH/Marina/MERRA2"
OUT_DIR = "/home/weiji/restart_exam/code/20260307merra2data"

os.makedirs(OUT_DIR, exist_ok=True)

OUT_DIR_CSV    = os.path.join(OUT_DIR, "merra2_directory_inventory.csv")
OUT_GROUP_CSV  = os.path.join(OUT_DIR, "merra2_nc_group_inventory.csv")
OUT_SAMPLE_CSV = os.path.join(OUT_DIR, "merra2_nc_sample_inventory.csv")
OUT_MISC_CSV   = os.path.join(OUT_DIR, "merra2_other_files_inventory.csv")

# ============================================================
# 1) 基础工具
# ============================================================
AUX_VAR_NAMES = {
    "bnds", "nbnds", "nv",
    "time_bnds", "lat_bnds", "lon_bnds", "lev_bnds",
    "latitude_bnds", "longitude_bnds"
}

def human_size(num_bytes):
    gb = num_bytes / (1024 ** 3)
    if gb >= 1:
        return f"{gb:.2f} GB"
    mb = num_bytes / (1024 ** 2)
    if mb >= 1:
        return f"{mb:.2f} MB"
    kb = num_bytes / 1024
    return f"{kb:.2f} KB"

def relpath_safe(path, root):
    return os.path.relpath(path, root)

def safe_get_attr(ds, key, default=""):
    val = ds.attrs.get(key, default)
    return "" if val is None else str(val)

def safe_open_metadata(ncfile):
    # 只读 metadata，不 decode 整个 dataset
    return xr.open_dataset(ncfile, decode_times=False)

def compact_join(items):
    items = [str(x).strip() for x in items if str(x).strip() != ""]
    return "; ".join(items)

# ============================================================
# 2) 变量、坐标、维度
# ============================================================
def classify_vars(ds):
    coord_names = list(ds.coords)
    data_var_names = list(ds.data_vars)

    core_vars = []
    aux_vars = []

    for v in data_var_names:
        lower = v.lower()

        if v in AUX_VAR_NAMES or lower.endswith("_bnds"):
            aux_vars.append(v)
            continue

        aux_vars_attr = str(ds[v].attrs.get("bounds", "")).strip()
        # 不因为有 bounds 属性就把主变量踢掉，只把明显 bnds 类变量归为辅助
        core_vars.append(v)

    return {
        "coords": coord_names,
        "core_vars": core_vars,
        "aux_vars": aux_vars,
    }

def get_dims(ds):
    return ", ".join([f"{k}={v}" for k, v in ds.sizes.items()])

def summarize_coord_metadata(ds):
    parts = []
    for cname in ds.coords:
        try:
            c = ds[cname]
            units = str(c.attrs.get("units", "")).strip()
            long_name = str(c.attrs.get("long_name", "")).strip()
            shape = "x".join([str(s) for s in c.shape]) if hasattr(c, "shape") else ""
            text = cname
            if shape:
                text += f"[{shape}]"
            if units:
                text += f" unit={units}"
            if long_name:
                text += f" long_name={long_name}"
            parts.append(text)
        except Exception:
            parts.append(cname)
    return compact_join(parts)

def summarize_var_attr(ds, var_names, attr_name):
    parts = []
    for v in var_names:
        try:
            val = str(ds[v].attrs.get(attr_name, "")).strip()
        except Exception:
            val = ""
        if val:
            parts.append(f"{v}={val}")
        else:
            parts.append(f"{v}=")
    return compact_join(parts)

# ============================================================
# 3) 分辨率与垂直信息
# ============================================================
def parse_dataresolution_string(s):
    if not s:
        return "", ""
    s = str(s).strip()
    m = re.match(r"^\s*([^()]+?)\s*(?:\((.*?)\))?\s*$", s)
    if not m:
        return s, ""
    horizontal = (m.group(1) or "").strip()
    vertical = (m.group(2) or "").strip()
    return horizontal, vertical

def infer_horizontal_resolution(ds):
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        hres, _ = parse_dataresolution_string(data_res)
        if hres:
            return hres

    lat_res = ds.attrs.get("LatitudeResolution", None)
    lon_res = ds.attrs.get("LongitudeResolution", None)
    if lat_res is not None and lon_res is not None:
        return f"{lat_res} x {lon_res}"

    try:
        lat_name = None
        lon_name = None
        for cand in ["lat", "latitude", "y"]:
            if cand in ds.coords:
                lat_name = cand
                break
        for cand in ["lon", "longitude", "x"]:
            if cand in ds.coords:
                lon_name = cand
                break

        if lat_name and lon_name and ds[lat_name].size > 1 and ds[lon_name].size > 1:
            lat_vals = ds[lat_name].values
            lon_vals = ds[lon_name].values
            dlat = abs(float(lat_vals[1] - lat_vals[0]))
            dlon = abs(float(lon_vals[1] - lon_vals[0]))
            return f"{dlat} x {dlon}"
    except Exception:
        pass

    return ""

def infer_vertical_info(ds):
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        _, vinfo = parse_dataresolution_string(data_res)
        if vinfo:
            return vinfo

    for zname in ["lev", "level", "plev", "isobaricInhPa", "isobaric", "pressure"]:
        if zname in ds.coords:
            try:
                nlev = ds[zname].size
                units = str(ds[zname].attrs.get("units", "")).strip()
                long_name = str(ds[zname].attrs.get("long_name", "")).strip()
                if units and long_name:
                    return f"{nlev} levels; {long_name}; {units}"
                elif units:
                    return f"{nlev} levels; {units}"
                else:
                    return f"{nlev} levels"
            except Exception:
                return "vertical coordinate present"

    return ""

# ============================================================
# 4) 时间信息：header + time axis
# ============================================================
def decode_numeric_time_point(value, units, calendar):
    if cftime is None:
        return ""
    try:
        out = cftime.num2date(value, units=units, calendar=calendar)
        return str(out)
    except Exception:
        return ""

def get_header_time_info(ds):
    return {
        "header_temporal_range": safe_get_attr(ds, "TemporalRange"),
        "header_begin_date": safe_get_attr(ds, "RangeBeginningDate"),
        "header_begin_time": safe_get_attr(ds, "RangeBeginningTime"),
        "header_end_date": safe_get_attr(ds, "RangeEndingDate"),
        "header_end_time": safe_get_attr(ds, "RangeEndingTime"),
    }

def get_time_axis_info(ds):
    row = {
        "time_coord_name": "",
        "time_len": "",
        "time_units": "",
        "time_calendar": "",
        "time_first_raw": "",
        "time_last_raw": "",
        "time_first_decoded": "",
        "time_last_decoded": "",
    }

    tname = None
    for cand in ["time", "Time", "valid_time"]:
        if cand in ds.coords:
            tname = cand
            break

    if tname is None:
        return row

    try:
        t = ds[tname]
        row["time_coord_name"] = tname
        row["time_len"] = str(t.size)
        row["time_units"] = str(t.attrs.get("units", "")).strip()
        row["time_calendar"] = str(t.attrs.get("calendar", "standard")).strip()

        if t.size > 0:
            first_val = t.isel({tname: 0}).values.item()
            last_val = t.isel({tname: -1}).values.item()

            row["time_first_raw"] = str(first_val)
            row["time_last_raw"] = str(last_val)

            # 如果已经是 datetime-like，直接转字符串
            if "datetime64" in str(type(first_val)).lower():
                row["time_first_decoded"] = str(first_val)
                row["time_last_decoded"] = str(last_val)
            else:
                units = row["time_units"]
                cal = row["time_calendar"] or "standard"
                if units:
                    row["time_first_decoded"] = decode_numeric_time_point(first_val, units, cal)
                    row["time_last_decoded"] = decode_numeric_time_point(last_val, units, cal)
    except Exception:
        pass

    return row

# ============================================================
# 5) 文件名时间 token 提取
# ============================================================
DATE_PATTERNS = [
    ("date10", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    ("date8", re.compile(r"(?<!\d)((?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    ("yearmon", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2]))(?!\d)")),
    ("year", re.compile(r"(?<!\d)((?:19|20)\d{2})(?!\d)")),
]

def normalize_time_token(token, kind):
    token = token.replace("_", "-").replace(".", "-")
    if kind == "date8":
        return f"{token[0:4]}-{token[4:6]}-{token[6:8]}"
    return token

def extract_time_tokens_no_overlap(text):
    accepted = []
    occupied = []

    def overlaps(span):
        s1, e1 = span
        for s2, e2 in occupied:
            if not (e1 <= s2 or e2 <= s1):
                return True
        return False

    for kind, pattern in DATE_PATTERNS:
        for m in pattern.finditer(text):
            span = m.span(1)
            if overlaps(span):
                continue
            raw = m.group(1)
            norm = normalize_time_token(raw, kind)
            accepted.append((span[0], span[1], kind, raw, norm))
            occupied.append(span)

    accepted.sort(key=lambda x: x[0])
    return accepted

def extract_time_tokens_from_filename(fname):
    base = os.path.basename(fname)
    records = extract_time_tokens_no_overlap(base)
    return [x[4] for x in records]

# ============================================================
# 6) 文件名 pattern 归一化
# ============================================================
def normalize_filename_pattern(fname):
    base = os.path.basename(fname)

    # 范围型
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<RANGE_YM_YM>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_YM_Y>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}-(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )

    # 单点日期
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE10>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE8>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<YEARMON>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?!\d)",
        "<YEAR>",
        base
    )

    return base

# ============================================================
# 7) sample file metadata 读取
# ============================================================
def summarize_sample_nc(ncfile):
    row = {
        "sample_status": "OK",
        "sample_title": "",
        "sample_short_name": "",
        "sample_long_name": "",
        "sample_frequency": "",
        "sample_doi": "",
        "sample_version": "",
        "sample_institution": "",
        "sample_comment": "",
        "sample_format": "",

        "sample_coords": "",
        "sample_core_vars": "",
        "sample_aux_vars": "",
        "sample_core_var_units": "",
        "sample_core_var_long_names": "",
        "sample_dimensions": "",
        "sample_horizontal_resolution": "",
        "sample_vertical_info": "",

        "header_temporal_range": "",
        "header_begin_date": "",
        "header_begin_time": "",
        "header_end_date": "",
        "header_end_time": "",

        "time_coord_name": "",
        "time_len": "",
        "time_units": "",
        "time_calendar": "",
        "time_first_raw": "",
        "time_last_raw": "",
        "time_first_decoded": "",
        "time_last_decoded": "",

        "missing_info_flags": "",
    }

    try:
        ds = safe_open_metadata(ncfile)

        var_info = classify_vars(ds)
        time_header = get_header_time_info(ds)
        time_axis = get_time_axis_info(ds)

        core_vars = var_info["core_vars"]

        row["sample_title"] = safe_get_attr(ds, "Title")
        row["sample_short_name"] = safe_get_attr(ds, "ShortName")
        row["sample_long_name"] = safe_get_attr(ds, "LongName")
        row["sample_frequency"] = safe_get_attr(ds, "frequency")
        row["sample_doi"] = safe_get_attr(ds, "identifier_product_doi")
        row["sample_version"] = safe_get_attr(ds, "VersionID")
        row["sample_institution"] = safe_get_attr(ds, "Institution")
        row["sample_comment"] = safe_get_attr(ds, "Comment")
        row["sample_format"] = safe_get_attr(ds, "Format")

        row["sample_coords"] = summarize_coord_metadata(ds)
        row["sample_core_vars"] = ", ".join(core_vars)
        row["sample_aux_vars"] = ", ".join(var_info["aux_vars"])
        row["sample_core_var_units"] = summarize_var_attr(ds, core_vars, "units")
        row["sample_core_var_long_names"] = summarize_var_attr(ds, core_vars, "long_name")
        row["sample_dimensions"] = get_dims(ds)
        row["sample_horizontal_resolution"] = infer_horizontal_resolution(ds)
        row["sample_vertical_info"] = infer_vertical_info(ds)

        row.update(time_header)
        row.update(time_axis)

        flags = []
        if row["sample_horizontal_resolution"] == "":
            flags.append("missing_horizontal_resolution")
        if row["sample_version"] == "":
            flags.append("missing_version")
        if row["sample_core_var_units"] == "":
            flags.append("missing_core_var_units")
        if row["header_temporal_range"] == "" and row["time_first_decoded"] == "":
            flags.append("missing_time_info")
        if row["sample_core_vars"] == "":
            flags.append("missing_core_vars")
        row["missing_info_flags"] = ", ".join(flags)

        ds.close()

    except Exception as e:
        row["sample_status"] = f"FAILED: {e}"

    return row

# ============================================================
# 8) group 分类与时间来源判断
# ============================================================
def has_time_like_pattern(pattern):
    markers = [
        "<DATE10>", "<DATE8>", "<YEARMON>", "<YEAR>",
        "<RANGE_YM_YM>", "<RANGE_YM_Y>", "<RANGE_Y_Y>"
    ]
    return any(m in pattern for m in markers)

def infer_product_type(pattern, n_files):
    lower = pattern.lower()

    if n_files > 1 and has_time_like_pattern(pattern):
        return "raw_file_family"

    if n_files > 1 and not has_time_like_pattern(pattern):
        return "multi_file_family"

    if ".zm" in lower or "_zm" in lower:
        return "processed_single_zm"
    if ".clim" in lower or "_clim" in lower:
        return "processed_single_clim"
    if ".pv" in lower or "_pv" in lower:
        return "processed_single_pv"
    if "_dm" in lower:
        return "processed_single_dm"

    return "single_product"

def infer_group_hint(pattern, n_files, sample_meta):
    lower = pattern.lower()
    tags = [infer_product_type(pattern, n_files)]

    if has_time_like_pattern(pattern):
        tags.append("time_in_filename")
    if ".zm" in lower or "_zm" in lower:
        tags.append("zonal_mean")
    if ".clim" in lower or "_clim" in lower:
        tags.append("climatology")
    if ".pv" in lower or "_pv" in lower:
        tags.append("PV_related")
    if "runmean" in lower:
        tags.append("running_mean")
    if "_dm" in lower:
        tags.append("daily_mean_or_dm")
    if "inst6" in lower:
        tags.append("inst6_source")
    if "noleap" in lower:
        tags.append("noleap")
    if "r144x96" in lower or "new_grid" in lower:
        tags.append("target_grid_or_regridded")

    freq = sample_meta.get("sample_frequency", "")
    if freq:
        tags.append(f"freq={freq}")

    return ", ".join(tags)

def choose_group_time_source(pattern, n_files, token_min, token_max, sample_meta):
    header_range = sample_meta.get("header_temporal_range", "")
    has_tokens = bool(token_min or token_max)

    if n_files > 1 and has_time_like_pattern(pattern) and has_tokens:
        return "filename_range"

    if n_files == 1 and header_range:
        return "header_temporal_range"

    if n_files == 1 and (sample_meta.get("header_begin_date") or sample_meta.get("header_end_date")):
        return "header_begin_end"

    if sample_meta.get("time_first_decoded") or sample_meta.get("time_last_decoded"):
        return "time_coordinate"

    if has_tokens:
        return "filename_range"

    return "unknown"

# ============================================================
# 9) 每个 group 选择样例文件：first / middle / last
# ============================================================
def select_sample_indices(n):
    if n <= 0:
        return []
    if n == 1:
        return [0]
    if n == 2:
        return [0, 1]
    return [0, n // 2, n - 1]

def summarize_group_consistency(sample_rows):
    """
    只做轻量一致性检查，不做最终解释
    """
    if len(sample_rows) <= 1:
        return "single_sample_only"

    checks = []

    def same(field):
        vals = [r.get(field, "") for r in sample_rows]
        return len(set(vals)) == 1

    checks.append(f"core_vars={'consistent' if same('sample_core_vars') else 'inconsistent'}")
    checks.append(f"units={'consistent' if same('sample_core_var_units') else 'inconsistent'}")
    checks.append(f"dims={'consistent' if same('sample_dimensions') else 'inconsistent'}")
    checks.append(f"hres={'consistent' if same('sample_horizontal_resolution') else 'inconsistent'}")
    checks.append(f"version={'consistent' if same('sample_version') else 'inconsistent'}")
    checks.append(f"time_len={'consistent' if same('time_len') else 'inconsistent'}")

    return ", ".join(checks)

# ============================================================
# 10) 扫描整个目录树
# ============================================================
dir_rows = []
group_rows = []
sample_rows = []
misc_rows = []

for current_dir, subdirs, files in os.walk(ROOT_DIR):
    subdirs.sort()
    files.sort()

    full_files = [os.path.join(current_dir, f) for f in files]
    nc_files = [f for f in full_files if f.endswith(".nc") or f.endswith(".nc4")]
    other_files = [f for f in full_files if not (f.endswith(".nc") or f.endswith(".nc4"))]

    # -------------------------
    # 目录级信息
    # -------------------------
    dir_rows.append({
        "directory": current_dir,
        "relative_directory": relpath_safe(current_dir, ROOT_DIR),
        "n_subdirs": len(subdirs),
        "subdirs": ", ".join(subdirs),
        "n_nc_files": len(nc_files),
        "n_other_files": len(other_files),
    })

    # -------------------------
    # 非 nc 文件
    # -------------------------
    for f in other_files:
        misc_rows.append({
            "directory": current_dir,
            "relative_directory": relpath_safe(current_dir, ROOT_DIR),
            "filename": os.path.basename(f),
            "size": human_size(os.path.getsize(f)),
            "extension": os.path.splitext(f)[1],
        })

    # -------------------------
    # 当前目录下的 nc 文件按 pattern 分组
    # -------------------------
    if len(nc_files) == 0:
        continue

    groups = {}
    for f in nc_files:
        patt = normalize_filename_pattern(os.path.basename(f))
        groups.setdefault(patt, []).append(f)

    for patt, flist in sorted(groups.items(), key=lambda x: x[0]):
        flist = sorted(flist)
        n_files = len(flist)

        # 文件名时间 token
        all_tokens = []
        for f in flist:
            all_tokens.extend(extract_time_tokens_from_filename(os.path.basename(f)))
        all_tokens = sorted(set(all_tokens))
        token_min = all_tokens[0] if len(all_tokens) > 0 else ""
        token_max = all_tokens[-1] if len(all_tokens) > 0 else ""

        # 大小统计
        size_list = [os.path.getsize(f) for f in flist]
        total_size_bytes = sum(size_list)
        min_size_bytes = min(size_list)
        max_size_bytes = max(size_list)

        # 抽样：first / middle / last
        sample_idxs = select_sample_indices(n_files)
        this_group_sample_rows = []

        for pos, idx in enumerate(sample_idxs, start=1):
            sf = flist[idx]
            meta = summarize_sample_nc(sf)

            sample_row = {
                "directory": current_dir,
                "relative_directory": relpath_safe(current_dir, ROOT_DIR),
                "file_pattern": patt,
                "sample_rank_in_group": pos,
                "sample_index": idx,
                "sample_file": sf,
                "sample_filename": os.path.basename(sf),
                "sample_file_size": human_size(os.path.getsize(sf)),
            }
            sample_row.update(meta)

            sample_rows.append(sample_row)
            this_group_sample_rows.append(sample_row)

        # group 级别用第一个 sample 作为代表
        rep = this_group_sample_rows[0] if len(this_group_sample_rows) > 0 else {}
        time_source = choose_group_time_source(
            pattern=patt,
            n_files=n_files,
            token_min=token_min,
            token_max=token_max,
            sample_meta=rep,
        )

        group_rows.append({
            "directory": current_dir,
            "relative_directory": relpath_safe(current_dir, ROOT_DIR),

            "file_pattern": patt,
            "n_files": n_files,
            "product_type": infer_product_type(patt, n_files),
            "group_hint": infer_group_hint(patt, n_files, rep),

            "first_file": os.path.basename(flist[0]),
            "last_file": os.path.basename(flist[-1]),
            "total_size": human_size(total_size_bytes),
            "min_file_size": human_size(min_size_bytes),
            "max_file_size": human_size(max_size_bytes),

            # 文件名时间
            "filename_token_min": token_min,
            "filename_token_max": token_max,
            "time_coverage_source_hint": time_source,

            # 代表 sample 的元信息
            "rep_sample_status": rep.get("sample_status", ""),
            "rep_sample_title": rep.get("sample_title", ""),
            "rep_sample_short_name": rep.get("sample_short_name", ""),
            "rep_sample_long_name": rep.get("sample_long_name", ""),
            "rep_sample_frequency": rep.get("sample_frequency", ""),
            "rep_sample_doi": rep.get("sample_doi", ""),
            "rep_sample_version": rep.get("sample_version", ""),
            "rep_sample_institution": rep.get("sample_institution", ""),

            "rep_sample_core_vars": rep.get("sample_core_vars", ""),
            "rep_sample_aux_vars": rep.get("sample_aux_vars", ""),
            "rep_sample_core_var_units": rep.get("sample_core_var_units", ""),
            "rep_sample_core_var_long_names": rep.get("sample_core_var_long_names", ""),
            "rep_sample_dimensions": rep.get("sample_dimensions", ""),
            "rep_sample_horizontal_resolution": rep.get("sample_horizontal_resolution", ""),
            "rep_sample_vertical_info": rep.get("sample_vertical_info", ""),
            "rep_sample_coords": rep.get("sample_coords", ""),

            # 代表 sample 的时间信息
            "rep_header_temporal_range": rep.get("header_temporal_range", ""),
            "rep_header_begin_date": rep.get("header_begin_date", ""),
            "rep_header_end_date": rep.get("header_end_date", ""),
            "rep_time_len": rep.get("time_len", ""),
            "rep_time_units": rep.get("time_units", ""),
            "rep_time_calendar": rep.get("time_calendar", ""),
            "rep_time_first_decoded": rep.get("time_first_decoded", ""),
            "rep_time_last_decoded": rep.get("time_last_decoded", ""),

            # 一致性检查
            "sample_consistency_check": summarize_group_consistency(this_group_sample_rows),

            # 缺失信息
            "rep_missing_info_flags": rep.get("missing_info_flags", ""),
        })

# ============================================================
# 11) 保存 raw inventory（不是最终表格）
# ============================================================
df_dirs = pd.DataFrame(dir_rows)
df_groups = pd.DataFrame(group_rows)
df_samples = pd.DataFrame(sample_rows)
df_misc = pd.DataFrame(misc_rows)

df_dirs.to_csv(OUT_DIR_CSV, index=False)
df_groups.to_csv(OUT_GROUP_CSV, index=False)
df_samples.to_csv(OUT_SAMPLE_CSV, index=False)
df_misc.to_csv(OUT_MISC_CSV, index=False)

# ============================================================
# 12) 终端摘要
# ============================================================
print("\nDone.")
print(f"Directories collected : {len(df_dirs)}")
print(f"NC groups collected   : {len(df_groups)}")
print(f"NC samples collected  : {len(df_samples)}")
print(f"Other files collected : {len(df_misc)}")

print("\nSaved raw inventories to:")
print(OUT_DIR_CSV)
print(OUT_GROUP_CSV)
print(OUT_SAMPLE_CSV)
print(OUT_MISC_CSV)

print("\nPreview of nc groups:")
preview_cols = [
    "relative_directory",
    "file_pattern",
    "n_files",
    "product_type",
    "filename_token_min",
    "filename_token_max",
    "time_coverage_source_hint",
    "rep_sample_core_vars",
    "rep_sample_horizontal_resolution",
    "rep_sample_vertical_info",
    "sample_consistency_check",
    "rep_missing_info_flags",
    "rep_sample_status",
]
if len(df_groups) > 0:
    print(df_groups[preview_cols].head(60).to_string(index=False))
else:
    print("No nc/nc4 files found.")

In [ ]:
import os
import re
import json
import pandas as pd
from collections import Counter, defaultdict

# ============================================================
# 0) 路径设置
# ============================================================
ROOT_DIR = "/pool/datos/reanalisis"
OUT_DIR = "/home/weiji/restart_exam/code/20260307merra2data"

os.makedirs(OUT_DIR, exist_ok=True)

OUT_DIR_CSV = os.path.join(OUT_DIR, "reanalisis_directory_inventory.csv")
OUT_PAT_CSV = os.path.join(OUT_DIR, "reanalisis_filename_pattern_inventory.csv")
OUT_TOP_TXT = os.path.join(OUT_DIR, "reanalisis_top_summary.txt")

# ============================================================
# 1) 基础工具
# ============================================================
def human_size(num_bytes):
    gb = num_bytes / (1024**3)
    if gb >= 1:
        return f"{gb:.2f} GB"
    mb = num_bytes / (1024**2)
    if mb >= 1:
        return f"{mb:.2f} MB"
    kb = num_bytes / 1024
    if kb >= 1:
        return f"{kb:.2f} KB"
    return f"{num_bytes} B"

def relpath_safe(path, root):
    return os.path.relpath(path, root)

def get_extension(fname):
    base = os.path.basename(fname)
    if base.endswith(".nc4"):
        return ".nc4"
    if base.endswith(".nc"):
        return ".nc"
    if base.endswith(".grib2"):
        return ".grib2"
    if base.endswith(".grib"):
        return ".grib"
    return os.path.splitext(base)[1].lower()

# ============================================================
# 2) 文件名归一化：用于看“命名模式”
# ============================================================
def normalize_filename_pattern(fname):
    base = os.path.basename(fname)

    # 范围型
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<RANGE_YM_YM>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_YM_Y>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}-(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>",
        base
    )

    # 日期 / 时间
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE10>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE8>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<YEARMON>",
        base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?!\d)",
        "<YEAR>",
        base
    )

    # 很长纯数字串
    base = re.sub(r"(?<!\d)\d{5,}(?!\d)", "<LONGNUM>", base)

    # 剩下的 2-4 位数字保守替换
    base = re.sub(r"(?<!\d)\d{2,4}(?!\d)", "<NUM>", base)

    return base

def extract_time_tokens_from_filename(fname):
    base = os.path.basename(fname)
    tokens = []

    patterns = [
        re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01]))(?!\d)"),
        re.compile(r"(?<!\d)((?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))(?!\d)"),
        re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2]))(?!\d)"),
        re.compile(r"(?<!\d)((?:19|20)\d{2})(?!\d)"),
    ]

    for pat in patterns:
        for m in pat.findall(base):
            t = m.replace("_", "-").replace(".", "-")
            if len(t) == 8 and t.isdigit():
                t = f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
            tokens.append(t)

    return sorted(set(tokens))

# ============================================================
# 3) 目录角色粗分类
# ============================================================
def guess_directory_role(n_files, n_subdirs, ext_counter, pattern_counter):
    exts = set(ext_counter.keys())
    has_nc = (".nc" in exts) or (".nc4" in exts)
    has_grib = (".grib" in exts) or (".grib2" in exts)

    many_files = n_files >= 500
    medium_files = n_files >= 30

    patterns = list(pattern_counter.keys())
    has_time_pattern = any(
        ("<DATE10>" in p) or ("<DATE8>" in p) or ("<YEARMON>" in p) or ("<YEAR>" in p)
        for p in patterns
    )

    if n_files == 0 and n_subdirs > 0:
        return "container_directory"
    if many_files and has_time_pattern and has_nc:
        return "likely_raw_netcdf_repository"
    if many_files and has_time_pattern and has_grib:
        return "likely_raw_grib_repository"
    if medium_files and has_time_pattern:
        return "time_partitioned_repository"
    if n_files <= 20 and has_nc:
        return "likely_processed_or_merged_products"
    if n_files > 0 and not has_time_pattern and has_nc:
        return "small_netcdf_collection"
    return "mixed_or_unclear"

# ============================================================
# 4) 扫描目录树
# ============================================================
dir_rows = []
pattern_rows = []

top_level_counter = Counter()
top_level_file_counter = Counter()
top_level_ext_counter = defaultdict(Counter)

for current_dir, subdirs, files in os.walk(ROOT_DIR):
    subdirs.sort()
    files.sort()

    rel_dir = relpath_safe(current_dir, ROOT_DIR)
    depth = 0 if rel_dir == "." else rel_dir.count(os.sep) + 1

    # 一级目录名
    first_part = rel_dir.split(os.sep)[0] if rel_dir != "." else "."

    full_files = [os.path.join(current_dir, f) for f in files]

    # 统计扩展名
    ext_counter = Counter()
    total_size_bytes = 0
    sample_files = []

    pattern_counter = Counter()
    pattern_to_files = defaultdict(list)
    pattern_to_tokens = defaultdict(list)

    for i, f in enumerate(full_files):
        fname = os.path.basename(f)
        ext = get_extension(fname)
        ext_counter[ext] += 1

        try:
            total_size_bytes += os.path.getsize(f)
        except Exception:
            pass

        if len(sample_files) < 5:
            sample_files.append(fname)

        patt = normalize_filename_pattern(fname)
        pattern_counter[patt] += 1
        if len(pattern_to_files[patt]) < 3:
            pattern_to_files[patt].append(fname)

        toks = extract_time_tokens_from_filename(fname)
        if len(toks) > 0:
            pattern_to_tokens[patt].extend(toks)

    role_guess = guess_directory_role(
        n_files=len(files),
        n_subdirs=len(subdirs),
        ext_counter=ext_counter,
        pattern_counter=pattern_counter,
    )

    dir_rows.append({
        "directory": current_dir,
        "relative_directory": rel_dir,
        "depth": depth,
        "top_level_group": first_part,
        "n_subdirs": len(subdirs),
        "subdirs": ", ".join(subdirs[:20]),
        "n_files": len(files),
        "total_size": human_size(total_size_bytes),
        "extensions": json.dumps(dict(ext_counter), ensure_ascii=False),
        "sample_files": " | ".join(sample_files),
        "n_unique_patterns": len(pattern_counter),
        "role_guess": role_guess,
    })

    # 每个目录内的 pattern 统计
    for patt, cnt in pattern_counter.most_common():
        token_list = sorted(set(pattern_to_tokens[patt]))
        pattern_rows.append({
            "directory": current_dir,
            "relative_directory": rel_dir,
            "depth": depth,
            "top_level_group": first_part,
            "file_pattern": patt,
            "n_files": cnt,
            "example_files": " | ".join(pattern_to_files[patt]),
            "filename_token_min": token_list[0] if len(token_list) > 0 else "",
            "filename_token_max": token_list[-1] if len(token_list) > 0 else "",
        })

    # 顶层统计
    top_level_counter[first_part] += 1
    top_level_file_counter[first_part] += len(files)
    for k, v in ext_counter.items():
        top_level_ext_counter[first_part][k] += v

# ============================================================
# 5) 保存
# ============================================================
df_dirs = pd.DataFrame(dir_rows)
df_patterns = pd.DataFrame(pattern_rows)

df_dirs.to_csv(OUT_DIR_CSV, index=False)
df_patterns.to_csv(OUT_PAT_CSV, index=False)

# ============================================================
# 6) 生成简短文本摘要
# ============================================================
lines = []
lines.append(f"ROOT_DIR: {ROOT_DIR}")
lines.append("")
lines.append(f"Directories scanned: {len(df_dirs)}")
lines.append(f"Pattern rows       : {len(df_patterns)}")
lines.append("")

lines.append("Top-level groups summary:")
for group in sorted(top_level_counter.keys()):
    lines.append(
        f"- {group}: directories={top_level_counter[group]}, "
        f"files={top_level_file_counter[group]}, "
        f"extensions={dict(top_level_ext_counter[group])}"
    )

lines.append("")
lines.append("Largest directories by file count:")
if len(df_dirs) > 0:
    tmp = df_dirs.sort_values("n_files", ascending=False).head(20)
    for _, r in tmp.iterrows():
        lines.append(
            f"- {r['relative_directory']}: n_files={r['n_files']}, "
            f"n_subdirs={r['n_subdirs']}, role_guess={r['role_guess']}"
        )

with open(OUT_TOP_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# ============================================================
# 7) 终端输出
# ============================================================
print("\nDone.")
print(f"Directories scanned : {len(df_dirs)}")
print(f"Pattern rows        : {len(df_patterns)}")
print("\nSaved to:")
print(OUT_DIR_CSV)
print(OUT_PAT_CSV)
print(OUT_TOP_TXT)

print("\nTop 30 directories by file count:")
if len(df_dirs) > 0:
    preview_cols = [
        "relative_directory",
        "depth",
        "n_subdirs",
        "n_files",
        "n_unique_patterns",
        "role_guess",
        "sample_files",
    ]
    print(df_dirs.sort_values("n_files", ascending=False)[preview_cols].head(30).to_string(index=False))
else:
    print("No directories found.")

In [ ]:
import os
import re
import json
import math
import pandas as pd
import xarray as xr

try:
    import cftime
except ImportError:
    cftime = None

try:
    import cfgrib  # noqa: F401
    CFGRIB_AVAILABLE = True
except Exception:
    CFGRIB_AVAILABLE = False


# ============================================================
# 0) 路径设置
# ============================================================
ROOT_DIR = "/pool/datos/reanalisis"
OUT_DIR = "/home/weiji/restart_exam/code/20260307merra2data"

# 第一阶段输出（如果存在，就直接利用）
DIR_INV_CSV = os.path.join(OUT_DIR, "reanalisis_directory_inventory.csv")
PAT_INV_CSV = os.path.join(OUT_DIR, "reanalisis_filename_pattern_inventory.csv")

# 第二阶段输出
OUT_TARGET_DIRS_CSV = os.path.join(OUT_DIR, "reanalisis_stage2_target_directories.csv")
OUT_GROUP_CSV       = os.path.join(OUT_DIR, "reanalisis_stage2_group_inventory.csv")
OUT_SAMPLE_CSV      = os.path.join(OUT_DIR, "reanalisis_stage2_sample_inventory.csv")
OUT_FAIL_CSV        = os.path.join(OUT_DIR, "reanalisis_stage2_failures.csv")

os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 1) 配置
# ============================================================
VALID_DATA_EXTS = {".nc", ".nc4", ".grib", ".grib2"}

# 若只想先看几个大类，可以改成：
# TOP_LEVEL_INCLUDE = ["merra2", "era5"]
TOP_LEVEL_INCLUDE = None

# 第一阶段 role_guess 的目录类型，默认都纳入
ROLE_INCLUDE = {
    "likely_raw_netcdf_repository",
    "likely_raw_grib_repository",
    "time_partitioned_repository",
    "small_netcdf_collection",
    "likely_processed_or_merged_products",
    "mixed_or_unclear",
}

# 抽样策略：
# n=1 -> [0]
# 2<=n<=12 -> [0, n-1]
# n>12 -> [0, n//2, n-1]
SAMPLE_MODE = "adaptive"

# 如果你想限制总读取量，可设置为整数；默认 None = 不限制
MAX_GROUPS_TOTAL = None


# ============================================================
# 2) 基础工具
# ============================================================
AUX_VAR_NAMES = {
    "bnds", "nbnds", "nv",
    "time_bnds", "lat_bnds", "lon_bnds", "lev_bnds",
    "latitude_bnds", "longitude_bnds"
}

def human_size(num_bytes):
    gb = num_bytes / (1024 ** 3)
    if gb >= 1:
        return f"{gb:.2f} GB"
    mb = num_bytes / (1024 ** 2)
    if mb >= 1:
        return f"{mb:.2f} MB"
    kb = num_bytes / 1024
    if kb >= 1:
        return f"{kb:.2f} KB"
    return f"{num_bytes} B"

def relpath_safe(path, root):
    return os.path.relpath(path, root)

def safe_get_attr(ds, key, default=""):
    val = ds.attrs.get(key, default)
    return "" if val is None else str(val)

def compact_join(items, sep="; "):
    items = [str(x).strip() for x in items if str(x).strip() != ""]
    return sep.join(items)

def get_extension(fname):
    base = os.path.basename(fname)
    if base.endswith(".nc4"):
        return ".nc4"
    if base.endswith(".nc"):
        return ".nc"
    if base.endswith(".grib2"):
        return ".grib2"
    if base.endswith(".grib"):
        return ".grib"
    return os.path.splitext(base)[1].lower()

def parse_extensions_json(s):
    if pd.isna(s) or str(s).strip() == "":
        return {}
    try:
        out = json.loads(s)
        if isinstance(out, dict):
            return out
        return {}
    except Exception:
        return {}

def choose_engine_by_ext(ext):
    if ext in {".nc", ".nc4"}:
        return "netcdf"
    if ext in {".grib", ".grib2"}:
        return "grib"
    return "unknown"

def select_sample_indices(n):
    if n <= 0:
        return []
    if SAMPLE_MODE != "adaptive":
        return [0] if n == 1 else [0, n - 1]

    if n == 1:
        return [0]
    if 2 <= n <= 12:
        return sorted(set([0, n - 1]))
    return sorted(set([0, n // 2, n - 1]))


# ============================================================
# 3) 文件名时间 token 与 pattern
# ============================================================
DATE_PATTERNS = [
    ("date10", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    ("date8",  re.compile(r"(?<!\d)((?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))(?!\d)")),
    ("yearmon", re.compile(r"(?<!\d)((?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2]))(?!\d)")),
    ("year", re.compile(r"(?<!\d)((?:19|20)\d{2})(?!\d)")),
]

def normalize_time_token(token, kind):
    token = token.replace("_", "-").replace(".", "-")
    if kind == "date8":
        return f"{token[0:4]}-{token[4:6]}-{token[6:8]}"
    return token

def extract_time_tokens_no_overlap(text):
    accepted = []
    occupied = []

    def overlaps(span):
        s1, e1 = span
        for s2, e2 in occupied:
            if not (e1 <= s2 or e2 <= s1):
                return True
        return False

    for kind, pattern in DATE_PATTERNS:
        for m in pattern.finditer(text):
            span = m.span(1)
            if overlaps(span):
                continue
            raw = m.group(1)
            norm = normalize_time_token(raw, kind)
            accepted.append((span[0], span[1], kind, raw, norm))
            occupied.append(span)

    accepted.sort(key=lambda x: x[0])
    return accepted

def extract_time_tokens_from_filename(fname):
    base = os.path.basename(fname)
    records = extract_time_tokens_no_overlap(base)
    return [x[4] for x in records]

def normalize_filename_pattern(fname):
    base = os.path.basename(fname)

    # 范围型
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<RANGE_YM_YM>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_YM_Y>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}\.(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}-(?:19|20)\d{2}(?!\d)",
        "<RANGE_Y_Y>", base
    )

    # 单点日期
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])[-_.](?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE10>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])(?!\d)",
        "<DATE8>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}[-_.](?:0[1-9]|1[0-2])(?!\d)",
        "<YEARMON>", base
    )
    base = re.sub(
        r"(?<!\d)(?:19|20)\d{2}(?!\d)",
        "<YEAR>", base
    )

    # 非日期数字
    base = re.sub(r"(?<!\d)\d{5,}(?!\d)", "<LONGNUM>", base)
    return base


# ============================================================
# 4) 元数据读取
# ============================================================
def open_dataset_metadata(path):
    ext = get_extension(path)
    engine_type = choose_engine_by_ext(ext)

    if engine_type == "netcdf":
        ds = xr.open_dataset(path, decode_times=False)
        return ds, "xarray-default"

    if engine_type == "grib":
        if not CFGRIB_AVAILABLE:
            raise RuntimeError("cfgrib not installed")
        ds = xr.open_dataset(
            path,
            engine="cfgrib",
            decode_times=False,
            backend_kwargs={"indexpath": ""}
        )
        return ds, "cfgrib"

    raise RuntimeError(f"unsupported extension: {ext}")

def classify_vars(ds):
    coord_names = list(ds.coords)
    data_var_names = list(ds.data_vars)

    core_vars = []
    aux_vars = []

    for v in data_var_names:
        lower = v.lower()
        if v in AUX_VAR_NAMES or lower.endswith("_bnds"):
            aux_vars.append(v)
        else:
            core_vars.append(v)

    return {
        "coords": coord_names,
        "core_vars": core_vars,
        "aux_vars": aux_vars,
    }

def get_dims(ds):
    return ", ".join([f"{k}={v}" for k, v in ds.sizes.items()])

def summarize_coord_metadata(ds):
    parts = []
    for cname in ds.coords:
        try:
            c = ds[cname]
            units = str(c.attrs.get("units", "")).strip()
            long_name = str(c.attrs.get("long_name", "")).strip()
            shape = "x".join([str(s) for s in c.shape]) if hasattr(c, "shape") else ""
            text = cname
            if shape:
                text += f"[{shape}]"
            if units:
                text += f" unit={units}"
            if long_name:
                text += f" long_name={long_name}"
            parts.append(text)
        except Exception:
            parts.append(cname)
    return compact_join(parts)

def summarize_var_attr(ds, var_names, attr_name):
    parts = []
    for v in var_names:
        try:
            val = str(ds[v].attrs.get(attr_name, "")).strip()
        except Exception:
            val = ""
        parts.append(f"{v}={val}")
    return compact_join(parts)

def parse_dataresolution_string(s):
    if not s:
        return "", ""
    s = str(s).strip()
    m = re.match(r"^\s*([^()]+?)\s*(?:\((.*?)\))?\s*$", s)
    if not m:
        return s, ""
    horizontal = (m.group(1) or "").strip()
    vertical = (m.group(2) or "").strip()
    return horizontal, vertical

def infer_horizontal_resolution(ds):
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        hres, _ = parse_dataresolution_string(data_res)
        if hres:
            return hres

    lat_res = ds.attrs.get("LatitudeResolution", None)
    lon_res = ds.attrs.get("LongitudeResolution", None)
    if lat_res is not None and lon_res is not None:
        return f"{lat_res} x {lon_res}"

    try:
        lat_name = None
        lon_name = None

        for cand in ["lat", "latitude", "y"]:
            if cand in ds.coords:
                lat_name = cand
                break
        for cand in ["lon", "longitude", "x"]:
            if cand in ds.coords:
                lon_name = cand
                break

        if lat_name and lon_name and ds[lat_name].size > 1 and ds[lon_name].size > 1:
            lat_vals = ds[lat_name].values
            lon_vals = ds[lon_name].values
            dlat = abs(float(lat_vals[1] - lat_vals[0]))
            dlon = abs(float(lon_vals[1] - lon_vals[0]))
            return f"{dlat} x {dlon}"
    except Exception:
        pass

    return ""

def infer_vertical_info(ds):
    data_res = ds.attrs.get("DataResolution", None)
    if data_res is not None:
        _, vinfo = parse_dataresolution_string(data_res)
        if vinfo:
            return vinfo

    for zname in ["lev", "level", "plev", "isobaricInhPa", "isobaric", "pressure"]:
        if zname in ds.coords:
            try:
                nlev = ds[zname].size
                units = str(ds[zname].attrs.get("units", "")).strip()
                long_name = str(ds[zname].attrs.get("long_name", "")).strip()
                if units and long_name:
                    return f"{nlev} levels; {long_name}; {units}"
                elif units:
                    return f"{nlev} levels; {units}"
                else:
                    return f"{nlev} levels"
            except Exception:
                return "vertical coordinate present"

    return ""

def get_header_time_info(ds):
    return {
        "header_temporal_range": safe_get_attr(ds, "TemporalRange"),
        "header_begin_date": safe_get_attr(ds, "RangeBeginningDate"),
        "header_begin_time": safe_get_attr(ds, "RangeBeginningTime"),
        "header_end_date": safe_get_attr(ds, "RangeEndingDate"),
        "header_end_time": safe_get_attr(ds, "RangeEndingTime"),
    }

def decode_numeric_time_point(value, units, calendar):
    if cftime is None:
        return ""
    try:
        out = cftime.num2date(value, units=units, calendar=calendar)
        return str(out)
    except Exception:
        return ""

def get_time_axis_info(ds):
    row = {
        "time_coord_name": "",
        "time_len": "",
        "time_units": "",
        "time_calendar": "",
        "time_first_raw": "",
        "time_last_raw": "",
        "time_first_decoded": "",
        "time_last_decoded": "",
    }

    tname = None
    for cand in ["time", "Time", "valid_time"]:
        if cand in ds.coords:
            tname = cand
            break

    if tname is None:
        return row

    try:
        t = ds[tname]
        row["time_coord_name"] = tname
        row["time_len"] = str(t.size)
        row["time_units"] = str(t.attrs.get("units", "")).strip()
        row["time_calendar"] = str(t.attrs.get("calendar", "standard")).strip()

        if t.size > 0:
            first_val = t.isel({tname: 0}).values.item()
            last_val = t.isel({tname: -1}).values.item()

            row["time_first_raw"] = str(first_val)
            row["time_last_raw"] = str(last_val)

            if "datetime64" in str(type(first_val)).lower():
                row["time_first_decoded"] = str(first_val)
                row["time_last_decoded"] = str(last_val)
            else:
                units = row["time_units"]
                cal = row["time_calendar"] or "standard"
                if units:
                    row["time_first_decoded"] = decode_numeric_time_point(first_val, units, cal)
                    row["time_last_decoded"] = decode_numeric_time_point(last_val, units, cal)
    except Exception:
        pass

    return row

def summarize_sample_file(path):
    ext = get_extension(path)
    row = {
        "sample_status": "OK",
        "sample_engine": "",
        "sample_ext": ext,
        "sample_title": "",
        "sample_short_name": "",
        "sample_long_name": "",
        "sample_frequency": "",
        "sample_doi": "",
        "sample_version": "",
        "sample_institution": "",
        "sample_source": "",
        "sample_conventions": "",
        "sample_format": "",
        "sample_comment": "",
        "sample_history": "",

        "sample_coords": "",
        "sample_core_vars": "",
        "sample_aux_vars": "",
        "sample_core_var_units": "",
        "sample_core_var_long_names": "",
        "sample_dimensions": "",
        "sample_horizontal_resolution": "",
        "sample_vertical_info": "",

        "header_temporal_range": "",
        "header_begin_date": "",
        "header_begin_time": "",
        "header_end_date": "",
        "header_end_time": "",

        "time_coord_name": "",
        "time_len": "",
        "time_units": "",
        "time_calendar": "",
        "time_first_raw": "",
        "time_last_raw": "",
        "time_first_decoded": "",
        "time_last_decoded": "",

        "missing_info_flags": "",
    }

    ds = None
    try:
        ds, used_engine = open_dataset_metadata(path)
        row["sample_engine"] = used_engine

        var_info = classify_vars(ds)
        core_vars = var_info["core_vars"]

        row["sample_title"] = safe_get_attr(ds, "Title")
        row["sample_short_name"] = safe_get_attr(ds, "ShortName")
        row["sample_long_name"] = safe_get_attr(ds, "LongName")
        row["sample_frequency"] = safe_get_attr(ds, "frequency")
        row["sample_doi"] = safe_get_attr(ds, "identifier_product_doi")
        row["sample_version"] = safe_get_attr(ds, "VersionID")
        row["sample_institution"] = safe_get_attr(ds, "Institution")
        row["sample_source"] = safe_get_attr(ds, "source")
        row["sample_conventions"] = safe_get_attr(ds, "Conventions")
        row["sample_format"] = safe_get_attr(ds, "Format")
        row["sample_comment"] = safe_get_attr(ds, "Comment")
        row["sample_history"] = safe_get_attr(ds, "History")

        row["sample_coords"] = summarize_coord_metadata(ds)
        row["sample_core_vars"] = ", ".join(core_vars)
        row["sample_aux_vars"] = ", ".join(var_info["aux_vars"])
        row["sample_core_var_units"] = summarize_var_attr(ds, core_vars, "units")
        row["sample_core_var_long_names"] = summarize_var_attr(ds, core_vars, "long_name")
        row["sample_dimensions"] = get_dims(ds)
        row["sample_horizontal_resolution"] = infer_horizontal_resolution(ds)
        row["sample_vertical_info"] = infer_vertical_info(ds)

        row.update(get_header_time_info(ds))
        row.update(get_time_axis_info(ds))

        flags = []
        if row["sample_horizontal_resolution"] == "":
            flags.append("missing_horizontal_resolution")
        if row["sample_version"] == "":
            flags.append("missing_version")
        if row["sample_core_var_units"] == "":
            flags.append("missing_core_var_units")
        if row["sample_core_vars"] == "":
            flags.append("missing_core_vars")
        if row["header_temporal_range"] == "" and row["time_first_decoded"] == "":
            flags.append("missing_time_info")
        row["missing_info_flags"] = ", ".join(flags)

    except Exception as e:
        row["sample_status"] = f"FAILED: {e}"

    finally:
        try:
            if ds is not None:
                ds.close()
        except Exception:
            pass

    return row


# ============================================================
# 5) 目标目录筛选
# ============================================================
def build_candidate_dirs_from_inventory():
    if not os.path.exists(DIR_INV_CSV):
        return None

    df = pd.read_csv(DIR_INV_CSV)

    keep_rows = []
    for _, r in df.iterrows():
        rel_dir = str(r.get("relative_directory", "."))
        role_guess = str(r.get("role_guess", ""))

        if TOP_LEVEL_INCLUDE is not None:
            first_part = rel_dir.split(os.sep)[0] if rel_dir != "." else "."
            if first_part not in TOP_LEVEL_INCLUDE:
                continue

        if role_guess not in ROLE_INCLUDE:
            continue

        ext_map = parse_extensions_json(r.get("extensions", ""))
        exts = set(ext_map.keys())

        if len(exts.intersection(VALID_DATA_EXTS)) == 0:
            continue

        keep_rows.append({
            "directory": r["directory"],
            "relative_directory": rel_dir,
            "depth": r.get("depth", ""),
            "n_files": r.get("n_files", 0),
            "extensions": r.get("extensions", ""),
            "role_guess": role_guess,
        })

    if len(keep_rows) == 0:
        return None

    out = pd.DataFrame(keep_rows).drop_duplicates(subset=["directory"]).sort_values(
        ["relative_directory"]
    )
    return out

def build_candidate_dirs_by_rescan():
    rows = []
    for current_dir, subdirs, files in os.walk(ROOT_DIR):
        rel_dir = relpath_safe(current_dir, ROOT_DIR)

        if TOP_LEVEL_INCLUDE is not None:
            first_part = rel_dir.split(os.sep)[0] if rel_dir != "." else "."
            if first_part not in TOP_LEVEL_INCLUDE:
                continue

        ext_counter = {}
        n_data_files = 0

        for f in files:
            ext = get_extension(f)
            ext_counter[ext] = ext_counter.get(ext, 0) + 1
            if ext in VALID_DATA_EXTS:
                n_data_files += 1

        if n_data_files > 0:
            rows.append({
                "directory": current_dir,
                "relative_directory": rel_dir,
                "depth": 0 if rel_dir == "." else rel_dir.count(os.sep) + 1,
                "n_files": len(files),
                "extensions": json.dumps(ext_counter, ensure_ascii=False),
                "role_guess": "rescan_candidate",
            })

    return pd.DataFrame(rows).sort_values(["relative_directory"])

def get_candidate_dirs():
    df = build_candidate_dirs_from_inventory()
    if df is not None and len(df) > 0:
        return df, "inventory_csv"
    return build_candidate_dirs_by_rescan(), "rescan"


# ============================================================
# 6) group-level 逻辑
# ============================================================
def has_time_like_pattern(pattern):
    markers = [
        "<DATE10>", "<DATE8>", "<YEARMON>", "<YEAR>",
        "<RANGE_YM_YM>", "<RANGE_YM_Y>", "<RANGE_Y_Y>"
    ]
    return any(m in pattern for m in markers)

def infer_product_type(pattern, n_files):
    lower = pattern.lower()

    if n_files > 1 and has_time_like_pattern(pattern):
        return "raw_file_family"
    if n_files > 1 and not has_time_like_pattern(pattern):
        return "multi_file_family"

    if ".zm" in lower or "_zm" in lower:
        return "processed_single_zm"
    if ".clim" in lower or "_clim" in lower:
        return "processed_single_clim"
    if ".pv" in lower or "_pv" in lower:
        return "processed_single_pv"
    if "_dm" in lower:
        return "processed_single_dm"

    return "single_product"

def choose_group_time_source(pattern, n_files, token_min, token_max, rep):
    has_tokens = bool(token_min or token_max)
    header_range = rep.get("header_temporal_range", "")

    if n_files > 1 and has_time_like_pattern(pattern) and has_tokens:
        return "filename_range"

    if n_files == 1 and header_range:
        return "header_temporal_range"

    if n_files == 1 and (rep.get("header_begin_date") or rep.get("header_end_date")):
        return "header_begin_end"

    if rep.get("time_first_decoded") or rep.get("time_last_decoded"):
        return "time_coordinate"

    if has_tokens:
        return "filename_range"

    return "unknown"

def summarize_group_consistency(sample_rows):
    if len(sample_rows) <= 1:
        return "single_sample_only"

    def same(field):
        vals = [r.get(field, "") for r in sample_rows]
        return len(set(vals)) == 1

    checks = [
        f"core_vars={'consistent' if same('sample_core_vars') else 'inconsistent'}",
        f"units={'consistent' if same('sample_core_var_units') else 'inconsistent'}",
        f"dims={'consistent' if same('sample_dimensions') else 'inconsistent'}",
        f"hres={'consistent' if same('sample_horizontal_resolution') else 'inconsistent'}",
        f"version={'consistent' if same('sample_version') else 'inconsistent'}",
        f"time_len={'consistent' if same('time_len') else 'inconsistent'}",
        f"engine={'consistent' if same('sample_engine') else 'inconsistent'}",
    ]
    return ", ".join(checks)


# ============================================================
# 7) 主流程
# ============================================================
candidate_dirs_df, candidate_source = get_candidate_dirs()
candidate_dirs_df.to_csv(OUT_TARGET_DIRS_CSV, index=False)

group_rows = []
sample_rows = []
fail_rows = []

group_counter = 0

for _, drow in candidate_dirs_df.iterrows():
    current_dir = drow["directory"]
    rel_dir = drow["relative_directory"]

    try:
        names = sorted(os.listdir(current_dir))
    except Exception as e:
        fail_rows.append({
            "relative_directory": rel_dir,
            "file_pattern": "",
            "sample_file": "",
            "stage": "listdir",
            "error": str(e),
        })
        continue

    data_files = []
    for name in names:
        path = os.path.join(current_dir, name)
        if not os.path.isfile(path):
            continue
        ext = get_extension(name)
        if ext in VALID_DATA_EXTS:
            data_files.append(path)

    if len(data_files) == 0:
        continue

    # 当前目录内按 pattern 分组
    groups = {}
    for f in data_files:
        patt = normalize_filename_pattern(os.path.basename(f))
        groups.setdefault(patt, []).append(f)

    for patt, flist in sorted(groups.items(), key=lambda x: x[0]):
        if MAX_GROUPS_TOTAL is not None and group_counter >= MAX_GROUPS_TOTAL:
            break

        flist = sorted(flist)
        n_files = len(flist)
        size_list = []

        all_tokens = []
        ext_counter = {}
        for f in flist:
            try:
                size_list.append(os.path.getsize(f))
            except Exception:
                pass
            ext = get_extension(f)
            ext_counter[ext] = ext_counter.get(ext, 0) + 1
            all_tokens.extend(extract_time_tokens_from_filename(os.path.basename(f)))

        all_tokens = sorted(set(all_tokens))
        token_min = all_tokens[0] if len(all_tokens) > 0 else ""
        token_max = all_tokens[-1] if len(all_tokens) > 0 else ""

        sample_idxs = select_sample_indices(n_files)
        this_group_sample_rows = []

        for rank, idx in enumerate(sample_idxs, start=1):
            sf = flist[idx]
            meta = summarize_sample_file(sf)

            srow = {
                "directory": current_dir,
                "relative_directory": rel_dir,
                "file_pattern": patt,
                "sample_rank_in_group": rank,
                "sample_index": idx,
                "sample_file": sf,
                "sample_filename": os.path.basename(sf),
                "sample_file_size": human_size(os.path.getsize(sf)) if os.path.exists(sf) else "",
            }
            srow.update(meta)
            sample_rows.append(srow)
            this_group_sample_rows.append(srow)

            if not str(meta.get("sample_status", "")).startswith("OK"):
                fail_rows.append({
                    "relative_directory": rel_dir,
                    "file_pattern": patt,
                    "sample_file": sf,
                    "stage": "open_metadata",
                    "error": meta.get("sample_status", ""),
                })

        rep = this_group_sample_rows[0] if len(this_group_sample_rows) > 0 else {}

        group_rows.append({
            "directory": current_dir,
            "relative_directory": rel_dir,
            "candidate_source": candidate_source,

            "file_pattern": patt,
            "n_files": n_files,
            "file_extensions": json.dumps(ext_counter, ensure_ascii=False),

            "first_file": os.path.basename(flist[0]),
            "last_file": os.path.basename(flist[-1]),
            "total_size": human_size(sum(size_list)) if len(size_list) > 0 else "",
            "min_file_size": human_size(min(size_list)) if len(size_list) > 0 else "",
            "max_file_size": human_size(max(size_list)) if len(size_list) > 0 else "",

            "filename_token_min": token_min,
            "filename_token_max": token_max,
            "time_coverage_source_hint": choose_group_time_source(
                patt, n_files, token_min, token_max, rep
            ),

            "product_type": infer_product_type(patt, n_files),

            "rep_sample_status": rep.get("sample_status", ""),
            "rep_sample_engine": rep.get("sample_engine", ""),
            "rep_sample_ext": rep.get("sample_ext", ""),
            "rep_sample_title": rep.get("sample_title", ""),
            "rep_sample_short_name": rep.get("sample_short_name", ""),
            "rep_sample_long_name": rep.get("sample_long_name", ""),
            "rep_sample_frequency": rep.get("sample_frequency", ""),
            "rep_sample_doi": rep.get("sample_doi", ""),
            "rep_sample_version": rep.get("sample_version", ""),
            "rep_sample_institution": rep.get("sample_institution", ""),
            "rep_sample_source": rep.get("sample_source", ""),
            "rep_sample_conventions": rep.get("sample_conventions", ""),

            "rep_sample_core_vars": rep.get("sample_core_vars", ""),
            "rep_sample_aux_vars": rep.get("sample_aux_vars", ""),
            "rep_sample_core_var_units": rep.get("sample_core_var_units", ""),
            "rep_sample_core_var_long_names": rep.get("sample_core_var_long_names", ""),
            "rep_sample_dimensions": rep.get("sample_dimensions", ""),
            "rep_sample_horizontal_resolution": rep.get("sample_horizontal_resolution", ""),
            "rep_sample_vertical_info": rep.get("sample_vertical_info", ""),
            "rep_sample_coords": rep.get("sample_coords", ""),

            "rep_header_temporal_range": rep.get("header_temporal_range", ""),
            "rep_header_begin_date": rep.get("header_begin_date", ""),
            "rep_header_end_date": rep.get("header_end_date", ""),
            "rep_time_coord_name": rep.get("time_coord_name", ""),
            "rep_time_len": rep.get("time_len", ""),
            "rep_time_units": rep.get("time_units", ""),
            "rep_time_calendar": rep.get("time_calendar", ""),
            "rep_time_first_decoded": rep.get("time_first_decoded", ""),
            "rep_time_last_decoded": rep.get("time_last_decoded", ""),

            "sample_consistency_check": summarize_group_consistency(this_group_sample_rows),
            "rep_missing_info_flags": rep.get("missing_info_flags", ""),
        })

        group_counter += 1

    if MAX_GROUPS_TOTAL is not None and group_counter >= MAX_GROUPS_TOTAL:
        break


# ============================================================
# 8) 保存
# ============================================================
df_groups = pd.DataFrame(group_rows)
df_samples = pd.DataFrame(sample_rows)
df_fails = pd.DataFrame(fail_rows)

df_groups.to_csv(OUT_GROUP_CSV, index=False)
df_samples.to_csv(OUT_SAMPLE_CSV, index=False)
df_fails.to_csv(OUT_FAIL_CSV, index=False)

print("\nDone.")
print(f"Candidate directories : {len(candidate_dirs_df)}")
print(f"Groups collected      : {len(df_groups)}")
print(f"Samples collected     : {len(df_samples)}")
print(f"Failures recorded     : {len(df_fails)}")

print("\nSaved to:")
print(OUT_TARGET_DIRS_CSV)
print(OUT_GROUP_CSV)
print(OUT_SAMPLE_CSV)
print(OUT_FAIL_CSV)

print("\nPreview of groups:")
preview_cols = [
    "relative_directory",
    "file_pattern",
    "n_files",
    "product_type",
    "filename_token_min",
    "filename_token_max",
    "rep_sample_core_vars",
    "rep_sample_core_var_units",
    "rep_sample_horizontal_resolution",
    "rep_sample_vertical_info",
    "rep_time_first_decoded",
    "rep_time_last_decoded",
    "sample_consistency_check",
    "rep_missing_info_flags",
    "rep_sample_status",
]
if len(df_groups) > 0:
    print(df_groups[preview_cols].head(80).to_string(index=False))
else:
    print("No group rows collected.")